In [49]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pickle

In [50]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144) 
  
# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

In [51]:
df_credit = pd.concat([X, y], axis=1)
df_credit.columns=["status", "duration", "credit_history", "purpose", "amount", 
                "savings", "employment_duration", "installment_rate",
                "personal_status_sex", "other_debtors",
                "present_residence", "property",
                "age", "other_installment_plans",
                "housing", "number_credits",
                "job", "people_liable", "telephone", "foreign_worker",
                "credit_risk"]
df_credit.dropna(inplace=True)

# Remap credit_risk: 1 (good) -> 0, 2 (bad) -> 1
df_credit['credit_risk'] = df_credit['credit_risk'].map({1: 0, 2: 1})

# Convert categorical string codes to integers
# Status: A11=1, A12=2, A13=3, A14=4
status_map = {'A11': 1, 'A12': 2, 'A13': 3, 'A14': 4}
# Credit history: A30=1, A31=2, A32=3, A33=4, A34=5
credit_history_map = {'A30': 1, 'A31': 2, 'A32': 3, 'A33': 4, 'A34': 5}
# Purpose: A40=1, A41=2, A42=3, A43=4, A44=5, A45=6, A46=7, A48=8, A49=9, A410=10
purpose_map = {'A40': 1, 'A41': 2, 'A42': 3, 'A43': 4, 'A44': 5, 'A45': 6, 'A46': 7, 'A48': 8, 'A49': 9, 'A410': 10}
# Savings: A61=1, A62=2, A63=3, A64=4, A65=5
savings_map = {'A61': 1, 'A62': 2, 'A63': 3, 'A64': 4, 'A65': 5}
# Employment duration: A71=1, A72=2, A73=3, A74=4, A75=5
employment_map = {'A71': 1, 'A72': 2, 'A73': 3, 'A74': 4, 'A75': 5}
# Personal status/sex: A91=1, A92=2, A93=3, A94=4, A95=5
personal_status_map = {'A91': 1, 'A92': 2, 'A93': 3, 'A94': 4, 'A95': 5}
# Other debtors: A101=1, A102=2, A103=3
other_debtors_map = {'A101': 1, 'A102': 2, 'A103': 3}
# Property: A121=1, A122=2, A123=3, A124=4
property_map = {'A121': 1, 'A122': 2, 'A123': 3, 'A124': 4}
# Other installment plans: A141=1, A142=2, A143=3
installment_plans_map = {'A141': 1, 'A142': 2, 'A143': 3}
# Housing: A151=1, A152=2, A153=3
housing_map = {'A151': 1, 'A152': 2, 'A153': 3}
# Job: A171=1, A172=2, A173=3, A174=4
job_map = {'A171': 1, 'A172': 2, 'A173': 3, 'A174': 4}
# Telephone: A191=1, A192=2
telephone_map = {'A191': 1, 'A192': 2}
# Foreign worker: A201=1, A202=2
foreign_worker_map = {'A201': 1, 'A202': 2}

# Apply mappings
df_credit['status'] = df_credit['status'].map(status_map)
df_credit['credit_history'] = df_credit['credit_history'].map(credit_history_map)
df_credit['purpose'] = df_credit['purpose'].map(purpose_map)
df_credit['savings'] = df_credit['savings'].map(savings_map)
df_credit['employment_duration'] = df_credit['employment_duration'].map(employment_map)
df_credit['personal_status_sex'] = df_credit['personal_status_sex'].map(personal_status_map)
df_credit['other_debtors'] = df_credit['other_debtors'].map(other_debtors_map)
df_credit['property'] = df_credit['property'].map(property_map)
df_credit['other_installment_plans'] = df_credit['other_installment_plans'].map(installment_plans_map)
df_credit['housing'] = df_credit['housing'].map(housing_map)
df_credit['job'] = df_credit['job'].map(job_map)
df_credit['telephone'] = df_credit['telephone'].map(telephone_map)
df_credit['foreign_worker'] = df_credit['foreign_worker'].map(foreign_worker_map)

# Ensure all values are numeric
df_credit = df_credit.astype(int)

In [52]:
df_credit.shape

(1000, 21)

In [35]:
# define protected attributes 
protected_attributes = ["age", "personal_status_sex", "foreign_worker"]

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

random_state = 1234
credit_train, credit_test = train_test_split(df_credit, test_size=0.2, random_state=random_state)

credit_train.to_parquet("credit_dataset/train_cleaned.parquet")
credit_test.to_parquet("credit_dataset/test_cleaned.parquet")

# Separate features and target
x_train = credit_train.drop("credit_risk", axis=1)
y_train = credit_train["credit_risk"]
x_test = credit_test.drop("credit_risk", axis=1)
y_test = credit_test["credit_risk"]

# Define numerical and categorical features
numerical_features = ["duration", "amount", "installment_rate", "present_residence", "age", "number_credits", "people_liable"]
ordinal_features = ["status", "credit_history", "savings", "employment_duration", "other_installment_plans", "housing", "job", "telephone", "foreign_worker"]
nominal_features = ["purpose", "personal_status_sex", "other_debtors", "property"]

In [54]:
# Create a preprocessed version suitable for Logistic Regression 

# One-hot encode nominal features
x_train_encoded = pd.get_dummies(x_train, columns=nominal_features, drop_first=False, dtype=int)
x_test_encoded = pd.get_dummies(x_test, columns=nominal_features, drop_first=False, dtype=int)

# ===== RENAME ONE-HOT ENCODED COLUMNS WITH MEANINGFUL NAMES =====
purpose_names = {1: 'new_car', 2: 'used_car', 3: 'furniture_equipment', 4: 'radio_television',
                 5: 'domestic_appliances', 6: 'repairs', 7: 'education', 8: 'retraining',
                 9: 'business', 10: 'others'}
personal_status_names = {1: 'male_divorced_separated', 2: 'female_divorced_separated_married',
                         3: 'male_single', 4: 'male_married_widowed', 5: 'female_single'}
other_debtors_names = {1: 'none', 2: 'co_applicant', 3: 'guarantor'}
property_names = {1: 'real_estate', 2: 'building_society_savings_life_insurance',
                  3: 'car_or_other', 4: 'unknown_no_property'}

# Create rename mapping
rename_map = {}
for col in x_train_encoded.columns:
    if col.startswith('purpose_'):
        val = int(col.split('_')[1])
        rename_map[col] = f'purpose_{purpose_names[val]}'
    elif col.startswith('personal_status_sex_'):
        val = int(col.split('_')[3])
        rename_map[col] = f'personal_status_{personal_status_names[val]}'
    elif col.startswith('other_debtors_'):
        val = int(col.split('_')[2])
        rename_map[col] = f'other_debtors_{other_debtors_names[val]}'
    elif col.startswith('property_'):
        val = int(col.split('_')[1])
        rename_map[col] = f'property_{property_names[val]}'

# Apply renaming
x_train_encoded = x_train_encoded.rename(columns=rename_map)
x_test_encoded = x_test_encoded.rename(columns=rename_map)

# Scale numerical features for logistic regression
scaler = StandardScaler()
x_train_scaled = x_train_encoded.copy()
x_test_scaled = x_test_encoded.copy()

x_train_scaled[numerical_features] = scaler.fit_transform(x_train_encoded[numerical_features])
x_test_scaled[numerical_features] = scaler.transform(x_test_encoded[numerical_features])

In [57]:
# random forest model 
rf_model = RandomForestClassifier(n_estimators=100, random_state=random_state)
rf_model.fit(x_train, y_train)

rf_pred = rf_model.predict(x_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")

# Save the model
with open('credit_dataset/RF.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

Accuracy: 71.00%


In [58]:
# random forest model 
rf_model = RandomForestClassifier(n_estimators=100, random_state=random_state)
rf_model.fit(x_train_scaled, y_train)

rf_pred = rf_model.predict(x_test_scaled)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")

# Save the model
with open('credit_dataset/RF_encoded.pkl', 'wb') as f:
    pickle.dump(rf_model, f)


Accuracy: 74.00%


In [59]:
# logistic regression model 
lr_model = LogisticRegression(max_iter=1000, random_state=random_state, solver='lbfgs')
lr_model.fit(x_train_scaled, y_train)

lr_pred = lr_model.predict(x_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)
print(f"Accuracy: {lr_accuracy * 100:.2f}%")

# Save the model
with open('credit_dataset/LR.pkl', 'wb') as f:
    pickle.dump(lr_model, f)



Accuracy: 77.00%


In [ ]:

feature_descriptions = [
    "Status of checking account (1: <0 DM, 2: 0-200 DM, 3: ≥200 DM/salary, 4: none)",
    "Duration of credit request in months",
    "Credit history rating (1: no credits-all paid, 2: all paid at bank, 3: existing paid till now, 4: past delays, 5: critical/other credits)",
    "Purpose of credit (1: new car, 2: used car, 3: furniture/equipment, 4: radio/TV, 5: domestic appliances, 6: repairs, 7: education, 8: retraining, 9: business, 10: others)",
    "Amount of credit requested in DM",
    "Savings account status (1: <100 DM, 2: 100-500 DM, 3: 500-1000 DM, 4: ≥1000 DM, 5: unknown/none)",
    "Employment duration (1: unemployed, 2: <1 year, 3: 1-4 years, 4: 4-7 years, 5: ≥7 years)",
    "Installment rate as percentage of disposable income",
    "Personal status and sex (1: male divorced/separated, 2: female divorced/separated/married, 3: male single, 4: male married/widowed, 5: female single)",
    "Other debtors/guarantors (1: none, 2: co-applicant, 3: guarantor)",
    "Years at present residence",
    "Property status (1: real estate, 2: building society savings/life insurance, 3: car/other, 4: unknown/none)",
    "Age in years",
    "Other installment plans (1: bank, 2: stores, 3: none)",
    "Housing situation (1: rent, 2: own, 3: for free)",
    "Number of existing credits at this bank",
    "Job type (1: unemployed/unskilled-non-resident, 2: unskilled-resident, 3: skilled/official, 4: management/self-employed/highly skilled)",
    "Number of people financially dependent",
    "Telephone (1: none, 2: yes registered)",
    "Foreign worker (1: yes, 2: no)"
]

feature_desc_df = pd.DataFrame({
    "feature_name": list(x_train.columns),
    "feature_average": x_train.mean().to_list(),
    "feature_desc": feature_descriptions,
})

# ===== FEATURE DESCRIPTIONS FOR x_train_scaled (37 encoded & scaled features) =====
feature_descriptions_scaled = [
    "Status: <0 DM [scaled]",
    "Duration in months [scaled]",
    "Credit history rating [scaled]",
    "Credit amount in DM [scaled]",
    "Savings status [scaled]",
    "Employment duration [scaled]",
    "Installment rate % [scaled]",
    "Years at residence [scaled]",
    "Age in years [scaled]",
    "Other installments [scaled]",
    "Housing status [scaled]",
    "Number of credits [scaled]",
    "Job type [scaled]",
    "Dependents [scaled]",
    "Telephone [scaled]",
    "Foreign worker [scaled]",
    "Purpose: new car",
    "Purpose: used car",
    "Purpose: furniture/equipment",
    "Purpose: radio/television",
    "Purpose: domestic appliances",
    "Purpose: repairs",
    "Purpose: education",
    "Purpose: retraining",
    "Purpose: business",
    "Purpose: others",
    "Personal status: male divorced/separated",
    "Personal status: female divorced/separated/married",
    "Personal status: male single",
    "Personal status: male married/widowed",
    "Other debtors: none",
    "Other debtors: co-applicant",
    "Other debtors: guarantor",
    "Property: real estate",
    "Property: building society/life insurance",
    "Property: car/other",
    "Property: unknown/none"
]

feature_desc_df_scaled = pd.DataFrame({
    "feature_name": list(x_train_scaled.columns),
    "feature_average": x_train_scaled.mean().to_list(),
    "feature_desc": feature_descriptions_scaled,
})

dataset_description = "The dataset contains information from the 1970s in Germany on a series of debtors that took a loan from the bank. It includes detailed categorical and numerical variables about their financial situation. Keep in mind that at the time Germany used Deutsche Marks (DM) with an average yearly salary of 10,000 to 20,000 DM"
target_description = "The target variable indicates whether the customer is a bad credit risk (1) or a good credit (0). We are predicting the bad credit class."
task_description = "Predict whether a new customer will be a bad credit risk"

# Dataset info for original features (x_train)
dataset_info = {
    "dataset_description": dataset_description,
    "target_description": target_description,
    "task_description": task_description,
    "feature_description": feature_desc_df
}

# Dataset info for encoded features (x_train_scaled)
dataset_info_scaled = {
    "dataset_description": dataset_description,
    "target_description": target_description,
    "task_description": task_description,
    "feature_description": feature_desc_df_scaled
}

# Save both versions
with open('credit_dataset/dataset_info', 'wb') as f:
    pickle.dump(dataset_info, f)

with open('credit_dataset/dataset_info_scaled', 'wb') as f:
    pickle.dump(dataset_info_scaled, f)

Feature descriptions updated successfully!

Original features (x_train): 20 features
Encoded features (x_train_scaled): 37 features
